# 04 Collect Yearly Data

Canonical data collection notebook for the refactored AMR workflow.
This notebook builds yearly pair counts, yearly background marginals, runs the unified LLM screen, and exports `yearly_amr_screened_pairs.parquet`.


In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.screening import (
    LLM_CLASSIFICATION_MAX_TOKENS,
    UNIFIED_AMR_INTEREST_SYSTEM_PROMPT,
    build_yearly_screened_pairs,
    prepare_yearly_candidate_pmids,
    run_unified_llm_filter,
)
from amr.llm import OpenAIConfig, OpenAILanguageModel
from amr.pipelines import build_and_print_collection_preflight
from amr.pipelines.collection import (
    collect_yearly_amr_dataset,
    collect_yearly_background_counts,
)
from amr.pipeline_config import cfg, cfg_path, load_pipeline_config


## Configuration
Set paths and run options. Use a real PubMed contact email and valid LLM credentials.


In [3]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)

ASSETS_DIR = cfg_path(PIPELINE_CONFIG, "paths.assets_dir", PROJECT_ROOT)
OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.output_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.parquet_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PARQUET_OUTPUT_DIR

PATHOGENS_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogens", PROJECT_ROOT)
PATHOGEN_PRIORITY_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogen_priority", PROJECT_ROOT)
MIN_PATHOGENIC_INDEX = cfg(PIPELINE_CONFIG, "collection.min_pathogenic_index")
ANTIBIOTIC_COARSENING_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_coarsening", PROJECT_ROOT)
ANTIBIOTIC_SYNONYM_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_synonyms", PROJECT_ROOT)
OUTPUT_YEARLY_AMR_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_candidates", PROJECT_ROOT)
OUTPUT_YEARLY_SCREEN_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_screened", PROJECT_ROOT)
OUTPUT_UNIFIED_DECISIONS_PATH = cfg_path(PIPELINE_CONFIG, "paths.unified_decisions", PROJECT_ROOT)
OUTPUT_PATHOGEN_BG_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogen_background", PROJECT_ROOT)
OUTPUT_ANTIBIOTIC_BG_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_background", PROJECT_ROOT)
INTERVAL_PMIDS_PATH = cfg_path(PIPELINE_CONFIG, "paths.interval_pmids", PROJECT_ROOT)
YEARLY_PAIR_PMID_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_pair_pmids", PROJECT_ROOT)
PUBMED_METADATA_CACHE_PATH = cfg_path(PIPELINE_CONFIG, "paths.pubmed_metadata_cache", PROJECT_ROOT)
ABSTRACT_CACHE_PATH = cfg_path(PIPELINE_CONFIG, "paths.abstract_cache", PROJECT_ROOT)
PUBMED_EMAIL = cfg(PIPELINE_CONFIG, "collection.pubmed_email")

# Date windows: TOTAL_* defines the canonical range; SLICE_* limits this run for partitioned/resumable work.
TOTAL_START_YEAR = cfg(PIPELINE_CONFIG, "collection.total_start_year")
TOTAL_END_YEAR = cfg(PIPELINE_CONFIG, "collection.total_end_year")
TOTAL_MIN_DATE = f"{TOTAL_START_YEAR}/01/01"
TOTAL_MAX_DATE = f"{TOTAL_END_YEAR}/12/31"
SLICE_START_YEAR = cfg(PIPELINE_CONFIG, "collection.slice_start_year")
SLICE_END_YEAR = cfg(PIPELINE_CONFIG, "collection.slice_end_year")
MIN_DATE = f"{SLICE_START_YEAR}/01/01"
MAX_DATE = f"{SLICE_END_YEAR}/12/31"
INTERVAL_YEAR_GAP = cfg(PIPELINE_CONFIG, "collection.interval_year_gap")

# Vocabulary limits: use None for full canonical runs; integers are useful for quick/resumable subsets.
NUM_PATHOGENS = cfg(PIPELINE_CONFIG, "collection.num_pathogens")
NUM_ANTIBIOTICS = cfg(PIPELINE_CONFIG, "collection.num_antibiotics")
EXTRA_AMR_KEYWORDS = cfg(PIPELINE_CONFIG, "collection.extra_amr_keywords", [])

# LLM decision modes: rerun_unified_decisions=True ignores active decisions but reuses cached abstracts.
RUN_LLM_FILTER = cfg(PIPELINE_CONFIG, "llm.run_filter")
RERUN_UNIFIED_LLM_DECISIONS = cfg(PIPELINE_CONFIG, "llm.rerun_unified_decisions")
MAX_LLM_CALLS = cfg(PIPELINE_CONFIG, "llm.max_calls")
UNIFIED_SYSTEM_PROMPT = UNIFIED_AMR_INTEREST_SYSTEM_PROMPT
LLM_MAX_WORKERS = cfg(PIPELINE_CONFIG, "llm.max_workers")
LLM_BATCH_SIZE = cfg(PIPELINE_CONFIG, "llm.batch_size")
FAILED_RETRY_PATH = cfg_path(PIPELINE_CONFIG, "paths.unified_failures", PROJECT_ROOT)
MAX_FAILURE_RETRIES = cfg(PIPELINE_CONFIG, "llm.max_failure_retries")
ABSTRACT_FETCH_BATCH_SIZE = cfg(PIPELINE_CONFIG, "llm.abstract_fetch_batch_size")
PREFETCH_ABSTRACTS = cfg(PIPELINE_CONFIG, "llm.prefetch_abstracts")
LLM_CATEGORY_BACKFILL_DECISIONS = cfg(PIPELINE_CONFIG, "llm.category_backfill_decisions")
LLM_RETRY_CACHED_FAILURES = bool(cfg(PIPELINE_CONFIG, "llm.retry_cached_failures", False))
LLM_INCLUDE_EXPLANATORY_TEXT = bool(cfg(PIPELINE_CONFIG, "llm.include_explanatory_text", False))
LLM_REASONING_EFFORT = cfg(PIPELINE_CONFIG, "llm.reasoning_effort", "minimal")
UNIFIED_PROGRESS_EVERY = cfg(PIPELINE_CONFIG, "llm.progress_every")
SCREENED_INCLUDE_DECISIONS = tuple(cfg(PIPELINE_CONFIG, "llm.screened_include_decisions", ["strict", "yes"]))

# PubMed throughput and background-count controls.
COLLECTION_VERBOSE = cfg(PIPELINE_CONFIG, "collection.collection_verbose")
COLLECTION_PROGRESS_EVERY = cfg(PIPELINE_CONFIG, "collection.collection_progress_every")
PAIR_FULL_FETCH_THRESHOLD = cfg(PIPELINE_CONFIG, "collection.pair_full_fetch_threshold")
COLLECTION_SAVE_EVERY = cfg(PIPELINE_CONFIG, "collection.save_every", 500)
COLLECTION_SUMMARY_BATCH_SIZE = cfg(PIPELINE_CONFIG, "collection.summary_batch_size", 1000)
COLLECTION_RETRIEVE_BATCH_SIZE = cfg(PIPELINE_CONFIG, "collection.retrieve_batch_size", 9000)
COLLECTION_MAX_WORKERS = cfg(PIPELINE_CONFIG, "collection.max_workers", 4)
COLLECTION_MAX_PENDING_FUTURES = cfg(PIPELINE_CONFIG, "collection.max_pending_futures", 8)
COLLECTION_PARTIAL_INTERVAL_MAX = cfg(PIPELINE_CONFIG, "collection.partial_interval_max", 1)
BACKGROUND_SCOPE = cfg(PIPELINE_CONFIG, "collection.background_scope")
BACKGROUND_FETCH_THRESHOLD = cfg(PIPELINE_CONFIG, "collection.background_fetch_threshold")
BACKGROUND_METADATA_BATCH_SIZE = cfg(PIPELINE_CONFIG, "collection.background_metadata_batch_size")


## Load Prepared Pathogen and Antibiotic Inputs



In [4]:
bacteria_synonym_df = pd.read_csv(PATHOGENS_PATH)
antibiotic_df = pd.read_csv(ANTIBIOTIC_COARSENING_PATH)
antibiotic_synonym_df = pd.read_csv(ANTIBIOTIC_SYNONYM_PATH)

if not PATHOGEN_PRIORITY_PATH.exists():
    raise FileNotFoundError(
        f"Pathogen priority file not found: {PATHOGEN_PRIORITY_PATH}. "
        "Run notebooks/01_prepare_inputs.ipynb first."
    )

pathogen_priority_df = pd.read_csv(PATHOGEN_PRIORITY_PATH)
required_priority_cols = {"CanonicalSpecies", "pathogenic_index"}
missing_priority_cols = required_priority_cols - set(pathogen_priority_df.columns)
if missing_priority_cols:
    raise KeyError(
        f"{PATHOGEN_PRIORITY_PATH} is missing required columns: {sorted(missing_priority_cols)}"
    )

pathogen_priority_df = pathogen_priority_df.copy()
pathogen_priority_df["CanonicalSpecies"] = pathogen_priority_df["CanonicalSpecies"].astype(str).str.strip()
pathogen_priority_df["pathogenic_index"] = pd.to_numeric(
    pathogen_priority_df["pathogenic_index"],
    errors="coerce",
)
pathogen_priority_df = pathogen_priority_df.dropna(subset=["CanonicalSpecies", "pathogenic_index"])
pathogen_priority_df = pathogen_priority_df.loc[
    pathogen_priority_df["pathogenic_index"] > float(MIN_PATHOGENIC_INDEX)
]
pathogen_priority_df = pathogen_priority_df.sort_values(
    ["pathogenic_index", "CanonicalSpecies"],
    ascending=[False, True],
).drop_duplicates(subset=["CanonicalSpecies"], keep="first")

all_pathogens = pathogen_priority_df["CanonicalSpecies"].tolist()
all_antibiotics = sorted(antibiotic_df["Antibiotic"].dropna().astype(str).str.strip().unique())

pathogens = all_pathogens if NUM_PATHOGENS is None else all_pathogens[: int(NUM_PATHOGENS)]
antibiotics = all_antibiotics if NUM_ANTIBIOTICS is None else all_antibiotics[: int(NUM_ANTIBIOTICS)]

print(f"Pathogen priority source: {PATHOGEN_PRIORITY_PATH}")
print(f"Full synonym lookup vocabulary: {bacteria_synonym_df['bacteria name'].nunique():,}")
print(
    f"Pathogenic-index pathogen vocabulary: {len(all_pathogens):,} "
    f"(MIN_PATHOGENIC_INDEX>{MIN_PATHOGENIC_INDEX})"
)
print(f"Antibiotic coarsening source: {ANTIBIOTIC_COARSENING_PATH}")
print(f"Antibiotic synonym source: {ANTIBIOTIC_SYNONYM_PATH} ({len(antibiotic_synonym_df):,} rows)")
print(f"Full antibiotic vocabulary: {len(all_antibiotics):,}")
print(f"Active pathogens this run: {len(pathogens):,} (NUM_PATHOGENS={NUM_PATHOGENS})")
print(f"Active antibiotics this run: {len(antibiotics):,} (NUM_ANTIBIOTICS={NUM_ANTIBIOTICS})")
print("Top active pathogens by pathogenic_index:")
display(
    pathogen_priority_df[["CanonicalSpecies", "pathogenic_index", "n_causation", "n_articles"]]
    .head(len(pathogens))
    .reset_index(drop=True)
)



Pathogen priority source: /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/counts_extended2.csv
Full synonym lookup vocabulary: 19,956
Pathogenic-index pathogen vocabulary: 915 (MIN_PATHOGENIC_INDEX>0.0)
Antibiotic coarsening source: /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/antibiotic_coarsening_lookup.csv
Antibiotic synonym source: /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/antibiotic_synonym_lookup.csv (422 rows)
Full antibiotic vocabulary: 245
Active pathogens this run: 400 (NUM_PATHOGENS=400)
Active antibiotics this run: 245 (NUM_ANTIBIOTICS=None)
Top active pathogens by pathogenic_index:


,CanonicalSpecies,pathogenic_index,n_causation,n_articles
0,Salmonella enterica,1.000000,554,24694
1,Mycobacterium tuberculosis,0.928144,299,20559
2,Clostridioides difficile,0.910833,298,17012
3,Haemophilus influenzae,0.860892,200,14537
4,Klebsiella pneumoniae,0.857160,176,15847
...,...,...,...,...
395,Clostridium paraputrificum,0.230388,13,32
396,Enterobacter cancerogenus,0.230388,13,34
397,Granulibacter bethesdensis,0.230388,13,16
398,Grimontia hollisae,0.230388,13,26


## Collect Yearly Pair Dataset and Background Marginals


In the next code cell, the notebook will automatically gather research paper IDs from PubMed for each pathogen-antibiotic combination in your selected year range, convert those papers into yearly counts, and then save the results to local files so future runs can continue from where you left off. It also collects yearly background publication totals for each pathogen and antibiotic, which are later used to calculate scientific interest in a fair, comparable way.


## Resume Preflight

Estimate how much collection work is already cached on disk before the resumable collection step starts.


In [5]:
preflight = build_and_print_collection_preflight(
    pathogens=pathogens,
    antibiotics=antibiotics,
    min_date=MIN_DATE,
    max_date=MAX_DATE,
    interval_year_gap=INTERVAL_YEAR_GAP,
    intervals_output_path=INTERVAL_PMIDS_PATH,
    yearly_pair_pmid_path=YEARLY_PAIR_PMID_PATH,
    yearly_output_path=OUTPUT_YEARLY_AMR_PATH,
    pathogen_bg_path=OUTPUT_PATHOGEN_BG_PATH,
    antibiotic_bg_path=OUTPUT_ANTIBIOTIC_BG_PATH,
    background_start_year=SLICE_START_YEAR,
    background_end_year=SLICE_END_YEAR,
    total_min_date=TOTAL_MIN_DATE,
    total_max_date=TOTAL_MAX_DATE,
    total_background_start_year=TOTAL_START_YEAR,
    total_background_end_year=TOTAL_END_YEAR,
)


COLLECTION PREFLIGHT

Existing yearly AMR span on disk: 2000 .. 2026
Existing yearly AMR rows on disk: 40,596
Existing yearly pair PMID rows on disk: 191,234

TOTAL WINDOW SUMMARY

Date window: 1970/01/01 .. 2026/12/31
Intervals in window: 12 -> [('1970/01/01', '1974/12/31'), ('1975/01/01', '1979/12/31'), ('1980/01/01', '1984/12/31'), ('1985/01/01', '1989/12/31'), ('1990/01/01', '1994/12/31'), ('1995/01/01', '1999/12/31'), ('2000/01/01', '2004/12/31'), ('2005/01/01', '2009/12/31'), ('2010/01/01', '2014/12/31'), ('2015/01/01', '2019/12/31'), ('2020/01/01', '2024/12/31'), ('2025/01/01', '2026/12/31')]
Interval cache coverage: 588,000 / 1,176,000 rows (50.00%)
Missing interval rows still to collect: 588,000
Cached interval breadth: 400 pathogens, 245 antibiotics, 98,000 unique pairs
Existing yearly AMR rows in window: 40,596
Existing yearly pair PMID rows in window: 191,234
Pathogen background cache: 4,725 / 22,800 rows (20.72%)
Missing pathogen background rows: 18,075 (79.28%)
Antibiotic

In [6]:
print("Auto-resume: collection stages will reuse saved outputs and continue missing work only.")

yearly_amr_df, yearly_pair_pmid_df, enriched_pair_pmid_df = collect_yearly_amr_dataset(
    pathogens=pathogens,
    antibiotics=antibiotics,
    bacteria_synonym_df=bacteria_synonym_df,
    antibiotic_synonym_df=antibiotic_synonym_df,
    min_date=MIN_DATE,
    max_date=MAX_DATE,
    email=PUBMED_EMAIL,
    interval_year_gap=INTERVAL_YEAR_GAP,
    intervals_output_path=INTERVAL_PMIDS_PATH,
    yearly_pair_pmid_output_path=YEARLY_PAIR_PMID_PATH,
    metadata_cache_path=PUBMED_METADATA_CACHE_PATH,
    output_path=OUTPUT_YEARLY_AMR_PATH,
    extra_keywords=EXTRA_AMR_KEYWORDS,
    full_fetch_threshold=PAIR_FULL_FETCH_THRESHOLD,
    save_every=COLLECTION_SAVE_EVERY,
    summary_batch_size=COLLECTION_SUMMARY_BATCH_SIZE,
    retrieve_batch_size=COLLECTION_RETRIEVE_BATCH_SIZE,
    max_workers=COLLECTION_MAX_WORKERS,
    max_pending_futures=COLLECTION_MAX_PENDING_FUTURES,
    partial_interval_max=COLLECTION_PARTIAL_INTERVAL_MAX,
    collection_verbose=COLLECTION_VERBOSE,
    collection_progress_every=COLLECTION_PROGRESS_EVERY,
)


Auto-resume: collection stages will reuse saved outputs and continue missing work only.
Collection status | pathogens 0/400 (0.0%) | current Salmonella enterica | antibiotics 0/245 (0.0%) | pairs 0/98000 (0.0%) | pair-years 0/2646000 (0.0%) | ETA --
Collection status | pathogens 0/400 (0.0%) | current Salmonella enterica | antibiotics 50/245 (20.4%) | pairs 50/98000 (0.1%) | pair-years 1350/2646000 (0.1%) | ETA 16s
Collection status | pathogens 0/400 (0.0%) | current Salmonella enterica | antibiotics 100/245 (40.8%) | pairs 100/98000 (0.1%) | pair-years 2700/2646000 (0.1%) | ETA 14s
Collection status | pathogens 0/400 (0.0%) | current Salmonella enterica | antibiotics 150/245 (61.2%) | pairs 150/98000 (0.2%) | pair-years 4050/2646000 (0.2%) | ETA 12s
Collection status | pathogens 0/400 (0.0%) | current Salmonella enterica | antibiotics 200/245 (81.6%) | pairs 200/98000 (0.2%) | pair-years 5400/2646000 (0.2%) | ETA 12s
Collection status | pathogens 1/400 (0.2%) | current Mycobacterium t

In [7]:
if BACKGROUND_SCOPE == "observed_entities":
    background_pathogens = sorted(yearly_amr_df["pathogen"].dropna().astype(str).unique())
    background_antibiotics = sorted(yearly_amr_df["antibiotic"].dropna().astype(str).unique())
elif BACKGROUND_SCOPE == "all_entities":
    background_pathogens = pathogens
    background_antibiotics = antibiotics
else:
    raise ValueError("BACKGROUND_SCOPE must be 'observed_entities' or 'all_entities'.")

print(
    f"Background scope={BACKGROUND_SCOPE}: "
    f"{len(background_pathogens):,} pathogens, {len(background_antibiotics):,} antibiotics"
)

pathogen_bg_df, antibiotic_bg_df = collect_yearly_background_counts(
    pathogens=background_pathogens,
    antibiotics=background_antibiotics,
    bacteria_synonym_df=bacteria_synonym_df,
    antibiotic_synonym_df=antibiotic_synonym_df,
    start_year=SLICE_START_YEAR,
    end_year=SLICE_END_YEAR,
    email=PUBMED_EMAIL,
    pathogen_output_path=OUTPUT_PATHOGEN_BG_PATH,
    antibiotic_output_path=OUTPUT_ANTIBIOTIC_BG_PATH,
    fetch_threshold=BACKGROUND_FETCH_THRESHOLD,
    metadata_batch_size=BACKGROUND_METADATA_BATCH_SIZE,
)

print(f"yearly AMR rows: {len(yearly_amr_df):,}")
print(f"yearly pair PMID rows: {len(yearly_pair_pmid_df):,}")
print(f"newly enriched pair PMID rows this run: {len(enriched_pair_pmid_df):,}")
print(f"pathogen bg rows: {len(pathogen_bg_df):,}")
print(f"antibiotic bg rows: {len(antibiotic_bg_df):,}")


Background scope=observed_entities: 271 pathogens, 198 antibiotics
Progress: 50/96 windows processed… | pathogen=Mycobacterium colombiense | years=2000-2026 | [ETA --]
[background_metadata:pathogen] fetching metadata for 80551 unique PMIDs from 87092 window-PMID assignments across 100 window(s).
[fetch_pub_metadata] fetching metadata for 80551 PMIDs in 81 batch(es)
[fetch_pub_metadata] starting batch 1/81 (1000 PMIDs) | ETA estimating | est finish --:--:--
[fetch_pub_metadata] starting batch 2/81 (1000 PMIDs) | ETA 0h 5m 16s | est finish 18:36:47
[fetch_pub_metadata] starting batch 3/81 (1000 PMIDs) | ETA 0h 5m 32s | est finish 18:37:07
Failed to parse Entrez summary (batch 3/81, attempt 1/3): IncompleteRead(68 bytes read). Retrying in 5s...
[fetch_pub_metadata] starting batch 4/81 (1000 PMIDs) | ETA 0h 9m 12s | est finish 18:41:00
[fetch_pub_metadata] starting batch 5/81 (1000 PMIDs) | ETA 0h 8m 8s | est finish 18:40:00
[fetch_pub_metadata] starting batch 6/81 (1000 PMIDs) | ETA 0h 7m

## Unified LLM Filtering
Single strict pass: AMR evidence + scientific-interest relevance.


In [8]:
#resume_df = pd.read_parquet(OUTPUT_UNIFIED_DECISIONS_PATH)
#unified_df = resume_df

In [9]:
candidate_df = prepare_yearly_candidate_pmids(yearly_amr_df)
candidate_df = candidate_df.loc[
    candidate_df["year"].between(SLICE_START_YEAR, SLICE_END_YEAR)
].reset_index(drop=True)
print(f"Yearly candidate PMIDs in slice [{SLICE_START_YEAR}, {SLICE_END_YEAR}]: {len(candidate_df):,}")

resume_df = None
# When rerun is False, existing compatible decisions are used for normal resume.
# retry_cached_failures=True reprocesses only rows previously saved as transient LLM failures.
# When rerun is True, old decisions and old retry-exhaustion state are ignored, while abstracts are reused.
auto_resume_unified = not RERUN_UNIFIED_LLM_DECISIONS
category_backfill_decisions = LLM_CATEGORY_BACKFILL_DECISIONS
active_failed_retry_path = FAILED_RETRY_PATH
if RERUN_UNIFIED_LLM_DECISIONS:
    active_failed_retry_path = PARQUET_OUTPUT_DIR / "yearly_amr_unified_failures_prompt_rerun.parquet"
    category_backfill_decisions = None
    print(
        "RERUN_UNIFIED_LLM_DECISIONS=True: ignoring existing unified decisions and rerunning LLM classification. "
        f"Cached abstracts at {ABSTRACT_CACHE_PATH} will still be reused."
    )
elif OUTPUT_UNIFIED_DECISIONS_PATH.exists():
    resume_df = pd.read_parquet(OUTPUT_UNIFIED_DECISIONS_PATH)
    retry_msg = " and cached LLM failures" if LLM_RETRY_CACHED_FAILURES else ""
    print(
        f"Found existing unified decisions: {len(resume_df):,} rows. "
        f"Missing candidates{retry_msg} will be processed automatically."
    )

if RUN_LLM_FILTER:
    llm = OpenAILanguageModel(OpenAIConfig(model=cfg(PIPELINE_CONFIG, "llm.model"), max_tokens=LLM_CLASSIFICATION_MAX_TOKENS, temperature=0))
    unified_df = run_unified_llm_filter(
        candidate_df,
        llm,
        email=PUBMED_EMAIL,
        system_prompt=UNIFIED_SYSTEM_PROMPT,
        output_path=str(OUTPUT_UNIFIED_DECISIONS_PATH),
        resume_from=resume_df,
        auto_resume_from_output=auto_resume_unified,
        llm_max_workers=LLM_MAX_WORKERS,
        llm_batch_size=LLM_BATCH_SIZE,
        abstract_fetch_batch_size=ABSTRACT_FETCH_BATCH_SIZE,
        prefetch_abstracts=PREFETCH_ABSTRACTS,
        failed_retry_path=str(active_failed_retry_path),
        max_failure_retries=MAX_FAILURE_RETRIES,
        retry_cached_llm_failures=LLM_RETRY_CACHED_FAILURES,
        abstract_cache_path=str(ABSTRACT_CACHE_PATH),
        max_llm_calls=MAX_LLM_CALLS,
        category_backfill_decisions=category_backfill_decisions,
        include_explanatory_text=LLM_INCLUDE_EXPLANATORY_TEXT,
        reasoning_effort=LLM_REASONING_EFFORT,
        progress_every=UNIFIED_PROGRESS_EVERY,
        verbose=1,
    )
else:
    if resume_df is None or resume_df.empty:
        raise FileNotFoundError(
            f"No unified decisions file found at {OUTPUT_UNIFIED_DECISIONS_PATH}. "
            "Enable RUN_LLM_FILTER to create it."
        )
    unified_df = resume_df
    print(f"Loaded unified decisions from {OUTPUT_UNIFIED_DECISIONS_PATH} ({len(unified_df):,} rows).")

print(unified_df["unified_decision"].value_counts(dropna=False))


Yearly candidate PMIDs in slice [2000, 2026]: 185,486
Found existing unified decisions: 182,607 rows. Missing candidates and cached LLM failures will be processed automatically.
Prefetched abstracts batch 1/2 (200 PMIDs).
Prefetched abstracts batch 2/2 (55 PMIDs).
[unified_filter] starting run for 2879 pending candidate rows (185486 input rows; skipped 182607 already complete)
Processed    200/2879   [strict:     5] [yes:   184] [no:    11] [uncertain:     0] | batch 0h 2m 3s | ETA 0h 27m 34s | est finish 19:07:27
Processed    400/2879   [strict:    13] [yes:   372] [no:    15] [uncertain:     0] | batch 0h 2m 2s | ETA 0h 25m 21s | est finish 19:07:15
Processed    600/2879   [strict:    17] [yes:   561] [no:    22] [uncertain:     0] | batch 0h 2m 12s | ETA 0h 23m 54s | est finish 19:08:00
Processed    800/2879   [strict:    24] [yes:   741] [no:    35] [uncertain:     0] | batch 0h 2m 1s | ETA 0h 21m 35s | est finish 19:07:43
Processed   1000/2879   [strict:    30] [yes:   908] [no:  

## Export Canonical Screened Yearly Dataset
Includes accepted LLM decisions `strict` + `yes`. `strict` means explicit pathogen-antibiotic evidence; `yes` means broader supported AMR panel/class/context evidence. `uncertain` is reserved for missing abstracts or technical failures and is not carried forward as AMR evidence.


In [10]:
SCREENED_INCLUDE_DECISIONS

('strict',)

In [13]:
# Reload persisted inputs so this cell can run independently after an interrupted LLM run.
yearly_amr_df = pd.read_parquet(OUTPUT_YEARLY_AMR_PATH)
unified_df = pd.read_parquet(OUTPUT_UNIFIED_DECISIONS_PATH)
print(f"Loaded yearly AMR rows from disk: {len(yearly_amr_df):,}")
print(f"Loaded unified decisions from disk: {len(unified_df):,}")

# If LLM decisions are force-rerun, rebuild screened yearly rows from only the new decisions.
# In normal resume mode, merging preserves already-screened years/pairs across partitioned runs.
yearly_screened_df = build_yearly_screened_pairs(
    yearly_amr_df,
    unified_df,
    include_decisions=SCREENED_INCLUDE_DECISIONS,
    output_path=str(OUTPUT_YEARLY_SCREEN_PATH),
    merge_with_existing_output=not RERUN_UNIFIED_LLM_DECISIONS,
)

print(f"Active pathogens this run: {len(pathogens):,}")
print(f"Observed pathogens in yearly AMR data: {yearly_amr_df['pathogen'].nunique():,}")
print(f"Screened yearly rows: {len(yearly_screened_df):,}")
print(f"Unique pairs: {yearly_screened_df[['pathogen','antibiotic']].drop_duplicates().shape[0]:,}")
print(f"Unique antibiotics: {yearly_screened_df['antibiotic'].nunique():,}")
print(f"Unique pathogens: {yearly_screened_df['pathogen'].nunique():,}")
print(f"Year span: {int(yearly_screened_df['year'].min())} - {int(yearly_screened_df['year'].max())}")
yearly_screened_df.head(10)



Loaded yearly AMR rows from disk: 40,596
Loaded unified decisions from disk: 185,486
Active pathogens this run: 400
Observed pathogens in yearly AMR data: 271
Screened yearly rows: 23,322
Unique pairs: 4,295
Unique antibiotics: 196
Unique pathogens: 184
Year span: 2000 - 2026


,pathogen,antibiotic,year,pmids,count
0,Achromobacter denitrificans,Amikacin,2026,[42414032],1
1,Achromobacter denitrificans,Aztreonam,2021,[33766786],1
2,Achromobacter denitrificans,Cefiderocol,2022,[35029278],1
3,Achromobacter denitrificans,Cefiderocol,2026,[41586494],1
4,Achromobacter denitrificans,Ceftazidime,2022,[35029278],1
5,Achromobacter denitrificans,Ceftazidime,2026,[41586494],1
6,Achromobacter denitrificans,Ceftazidime/avibactam,2026,[41586494],1
7,Achromobacter denitrificans,Ciprofloxacin,2013,[22944724],1
8,Achromobacter denitrificans,Ciprofloxacin,2021,[33766786],1
9,Achromobacter denitrificans,Ciprofloxacin,2022,[35029278],1
